#📚 文系大学生による『実践データ分析100本ノック＆LLM活用20本ノック』学習ログ
**著： 岡野 菜月（四国大学文学部 / 28卒）**

**GitHub： https://github.com/NatsukiOkano**

##📝 このノートブックの概要
＊ 書籍「Python 実践データ分析100本ノック 第3版」（秀和システム新社刊）の学習過程を記録したものです。

＊ 最大の特徴は、「文学部の視点でIT・データ分析を再解釈する」という試みにあります。

＊ 非情報系の学習者が直面する「専門用語の壁」を取り払うため、すべての処理に対し、論理的かつ直感的な「完全独自解説」を記述しています。

##⚠️ 注意事項
- **著作権保護の観点から、データセットおよび実行結果は同梱しておりません。**

### 教材
下山輝昌・松田雄馬・三木孝行 著「Python 実践データ分析100本ノック 第3版」（秀和システム新社、2025）

# 4章 顧客の全体像を把握する１０本ノック

引き続き、スポーツジムの会員データを使って顧客の行動を分析していきます。  
３章で顧客の全体像を把握しました。  
ここからは、機械学習を用いて顧客のグループ化や顧客の利用予測行なっていきましょう。  
ここでは、教師なし学習、教師あり学習の回帰を取り扱います。

### ノック31：データを読み込んで確認しよう

In [ ]:
import pandas as pd
uselog = pd.read_csv('use_log.csv')
uselog.isnull().sum()

In [ ]:
customer = pd.read_csv('customer_join.csv')
customer.isnull().sum()

### ノック32：クラスタリングで顧客をグループ化しよう

In [ ]:
#分類：大量に読み込ませたデータを特定のクラスに振り分ける「教師あり学習」の手法。
#回帰：大量に読み込ませたデータから数値を予測する「教師あり学習」の手法。
#クラスタリング：大量に読み込ませたデータを類似性に基づいて自動的にグループ化する「教師なし学習」の手法。
#グルーピング：グループ化。
#強化学習：経験から学ぶAI。試行錯誤を通じて、報酬を最大化するための最適な行動を自ら学習する。

In [ ]:
customer_clustering = customer[['mean', 'median', 'max', 'min', 'membership_period']]
customer_clustering.head()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

#from sklearn.cluster：『ライブラリ（sklearn）』に格納される『モジュール（cluster）』を指定する。
#import KMeans：『モジュール（cluster）』に格納される『クラス（KMeans）』を自分のプログラムに導入する。

#ライブラリ（sklearn）：「教師あり学習」「教師なし学習」に100%特化したライブラリ。※強化学習は非対応。
#モジュール（cluster）：クラスタリングのプログラムを格納したファイル。
#モジュール（preprocessing）：AIが扱いやすいようにデータの単位や型を均一にするプログラムを格納したファイル。
#クラス（KMeans）：グループの平均値を探すAI。グループ全体のバラつき(標準偏差)を最小限にする。K＝グループ数。
#クラス（StandardScaler）：補正装置。バラついたデータの単位や型を標準化して均一に整える。※AIではない。

#パッケージ：類似性のあるモジュールを格納したフォルダ。
#モジュール：プログラムを再利用するために、コードを格納したファイル。
#クラス：データ型。

In [ ]:
customer_clustering_sc = sc.fit_transform(customer_clustering)

#sc.fit_transform：補正装置（StandardScaler）に、読み込ませたデータを一気に補正してもらう。

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=0)
clusters = kmeans.fit(customer_clustering_sc)
customer_clustering = customer_clustering.assign(cluster = clusters.labels_)

#n_clusters：グループの平均値を探すAI（KMeans）にクラスタ数を指定する。
#※クラスタ数：分類するグループ(グループ化)の個数。

#random_state：AIのランダム性(気の変わり)を抑える。
#random_state=0：毎回、「0」が名前に設定されたデータを出力。

#モデル.fit：AIの学習を開始する。

#df.assign：データフレームに新しい列を付け加える。※行を付け加えることはできない。
#df.assign(cluster)：新しい列の名前(列ラベル)「cluster」。
#モデル.labels_：AIに「学習結果(グループ化した結果)を格納したグループ番号」を自動で入力してもらう。
#df.assign(cluster = clusters.labels_)：AIに「cluster」という名前の新しい列に「学習結果(グループ化した結果)を格納したグループ番号」を自動で入力してもらう。

In [ ]:
print(customer_clustering['cluster'].unique())
customer_clustering.head()

#df['列名'].unique：重複を削ぎ落した「純粋な種類の一覧」を1行で表示させる。

### ノック33：クラスタリング結果を分析しよう

In [ ]:
customer_clustering.columns = ['月内平均値','月内中央値','月内最大値','月内最小値','会員期間','cluster']
customer_clustering.groupby('cluster').count()

#groupby：データを同じ仲間ごとにグループ分けする。

In [ ]:
customer_clustering.groupby('cluster').mean()

### ノック34：クラスタリング結果を可視化してみよう

In [ ]:
#プロット：グラフ化する。
#次元削減：教師なし学習の1つ。データの列(変数)が多いとき、情報の損失を最小限に抑えながら複数の列(変数)を合成して新しい列(変数)を作り出すことで、全体のデータ量を簡潔にする。
#主成分分析(PCA)：次元削減を実行するAI。

In [ ]:
from sklearn.decomposition import PCA

#モジュール（decomposition）：次元削減するファイルを集めたフォルダ。
#クラス（PCA)：主成分分析AI。次元削減を実行する。

In [ ]:
X = customer_clustering_sc
pca = PCA(n_components=2)
pca.fit(X)
x_pca = pca.transform(X)
pca_df = pd.DataFrame(x_pca)
pca_df['cluster'] = customer_clustering['cluster']
pca_df.head()

#n_clusters：グループの平均値を探すAI（KMeans）にクラスタ数を指定する。※クラスタ数：分類するグループ(グループ化)の個数。
#n_components：主成分分析AI（PCA）に「最終的に合成して作り出す新しい列の数」を指定する。

#モデル.fit：AIの学習を開始する。
#モデル.transform：AIの学習をデータに適用する。
#pd.DataFrame()：データフレーム(表)を作成する。

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
for i in customer_clustering['cluster'].unique():
    tmp = pca_df.loc[pca_df['cluster']==i]
    plt.scatter(tmp[0], tmp[1])

#matplotlib：全ての工程を自らの手で制御する「プロ仕様版」グラフ作成ツール。
#matplotlib.pyplot：複雑な工程を自動化した「簡易操作版」グラフ作成ツール。

#マジックコマンド（% , %%）：プログラムの外側の設定を変更する。
# %（ラインマジック）：その一行だけに効く命令。
# %%（セルマジック）：そのセル全体に効く命令。

#【よく使うマジックコマンド（↓）】
#%matplotlib inline（グラフをノートブック内に直接表示する。）
#%timeit（その行のコードの実行速度を計測する）
#%pwd（現在のフォルダにあるファイル一覧を表示する。）
#%run（外部のPythonファイルをノートブック上で実行する。）

#df.unique：重複を削ぎ落した「純粋な種類の一覧」を1行で表示させる。
#loc[行ラベル,列ラベル]：特定の条件に当てはまるデータだけを書き換える。

#「~」：否定演算子。「NOT（～ではない）」を意味する記号。※flg_is_null：変数。~flg_is_null：flg_is_nullの反対。
#「=」：代入演算子。代入。例）x = 100：xに100を代入。→これ以降、xは100として扱われる。
#「==」：比較演算子。比較。(判定)　例）x == 100：xの中身は100と同じ？→もしxが100ならTrueになる。

#plt.plot（折れ線グラフ）
#plt.bar（棒グラフ）
#plt.scatter（散布図）
#plt.hist（ヒストグラム）

#plt.scatter(x軸(横軸), y軸(縦軸))：散布図。主成分分析。※x軸(横軸) = 第1主成分, y軸(縦軸) = 第2主成分
#tmp[0]：1番目の新しい列（第1主成分）
#tmp[1]：2番目の新しい列（第2主成分）


### ノック35：クラスタリング結果をもとに退会顧客の傾向を把握しよう

In [ ]:
customer_clustering.head()

In [ ]:
customer_clustering = pd.concat([customer_clustering, customer], axis=1)
customer_clustering.groupby(['cluster','is_deleted'],as_index=False).count()[['cluster','is_deleted','customer_id']]

#pd.concat：結合。同じ形式の複数の表を、上下にレゴブロックのように積み上げる。
#axis：処理する向き　axis=0：縦（列）　axis=1：横（行）

#groupby：データを同じ仲間ごとにグループ分けする。
#as_index：groupby()の引数。グループ化した項目を、新しい表の「見出し(インデックス)」にするか、「普通のデータ(列)」のままにするか決める。
#as_index = True：グループ名が、新しい表の「見出し(インデックス)」に。
#as_index = False：グループ名がそのまま「普通のデータ(列)」として残る。

In [ ]:
customer_clustering.groupby(['cluster','routine_flg'],as_index=False).count()[['cluster','routine_flg','customer_id']]

### ノック36：翌月の利用回数予測を行うためのデータ準備をしよう

In [ ]:
uselog.head()

In [ ]:
uselog['usedate'] = pd.to_datetime(uselog['usedate'])
uselog['年月'] = uselog['usedate'].dt.strftime('%Y%m')
uselog_months = uselog.groupby(['年月','customer_id'],as_index=False).count()
uselog_months.rename(columns={'log_id':'count'}, inplace=True)
del uselog_months['usedate']
uselog_months.head()

#pd.to_datetime：「文字（object）」として読み込まれた日付を、計算や加工ができる『日付専用データ（datetime）』に作り変える命令。
#to：～へ変換する。

#dt.strftime：「dt」datetimeの略。「strftime」時間を、形式を整えた文字にする。「string format time」の略。string（文字）format（形式を整える）time（時間）
#決まり文句：「%Y」西暦（4桁）,「%y」西暦（2桁）,「%m」月（2桁）,「%d」日（2桁）,「%H」時（2桁）,「%M」分（2桁）
#例：「%Y」2025,「%y」25,「%m」02,「%d」03,「%H」15,「%M」31

#groupby：データを同じ仲間ごとにグループ分けする。
#as_index：groupby()の引数。グループ化した項目を、新しい表の「見出し(インデックス)」にするか、「普通のデータ(列)」のままにするか決める。
#as_index = True：グループ名が、新しい表の「見出し(インデックス)」に。
#as_index = False：グループ名がそのまま「普通のデータ(列)」として残る。

#rename：列ラベル(columns)や行ラベル(index)を書き換える。
#columns：「column(列)」の複数形。

#inplace：「『元のデータ』そのものに修正を加える」か、「『コピーした新しいデータ』に修正を加えてユーザーに渡す」か決める。
#inplace = True：『元のデータ』そのものに修正を加える。※ユーザーに渡さない。
#inplace = False：『コピーした新しいデータ』に修正を加えてユーザーに渡す。

#del：データ(オブジェクト)そのものを抹消するためのPython標準の命令。

In [ ]:
year_months = list(uselog_months['年月'].unique())
predict_data = pd.DataFrame()

#list()：抽出したデータをPython標準の「リスト型」に変換する。
#df['列名'].unique：重複を削ぎ落した「純粋な種類の一覧」を1行で表示させる。
#pd.DataFrame：データフレーム(表)を作成する。

In [ ]:
for i in range(6, len(year_months)):
    tmp = uselog_months.loc[uselog_months['年月']==year_months[i]].copy()
    tmp.rename(columns={'count':'count_pred'}, inplace=True)
    for j in range(1, 7):
        tmp_before = uselog_months.loc[uselog_months['年月']==year_months[i-j]].copy()
        del tmp_before['年月']
        tmp_before.rename(columns={'count':f'count_{j-1}'}, inplace=True)
        tmp = pd.merge(tmp, tmp_before, on='customer_id', how='left')
    predict_data = pd.concat([predict_data, tmp], ignore_index=True)
predict_data.head()

#for i in range：「繰り返し処理」の基本セット
  # for：～の間、繰り返す。
  # i：今、何回目か数えるための変数。
  # in range()：繰り返す回数。

#len：データの件数を数える。Excelのcount関数と同じ役割。
#tmp：変数
#tmp_before：変数

#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[]：整数で指定。整数を使って条件に合うデータを探す。

#df.copy()：複製。コピーして新しいデータフレーム(表)を作る。

#.format() = f-string（f文字列）：文字と変数を繋ぐ。
#【コードの書き方の違い(例文)↓】
#.format()：'{}の担当は{}です'.format(month,name)
#f-string（f文字列）：f'{month}の担当は{name}です'

#pd.concat()：結合。同じ形式(列名)のデータフレーム(表)を、縦(行方向)に結合する。
#ignore_index：0から順番に行番号(index)を新しく振り直す。
#ignore_index=True：0から順番に行番号(index)を新しく振り直す。
#ignore_index=False：0から順番に行番号(index)を新しく振り直さず、元の行番号(index)を使う。

#pd.merge()：結合。正規化されてバラバラになったデータを、分析しやすいように「逆正規化（合体）」させる。
#pd.merge(左の表,右の表, left_on=, right_on=, how=)
#left_on：左側の表(テーブル)の使用する列名を指定。
#right_on：右側の表(テーブル)の使用する列名を指定。
#※left_on, right_on はそれぞれの表(テーブル)で使う列名が異なるときに使う。
#how：表(テーブル)の結合方法を指定。（※よく使う結合方法↓）
#how='left'：左の表を主役にする。右に一致するデータがなくても、左のデータはすべて残す。
#how='right'：右の表を主役にする。左に一致するデータがなくても、右のデータはすべて残す。
#how='inner'：両方の表にあるデータだけを残す。どちらか片方にしかないデータは消える。
#how='outer'：両方の表のデータをすべて残す。一致しない部分は「欠損値(NaN)」で埋める。

In [ ]:
list(uselog['年月'].unique())
#Python標準の「リスト型」
  #データを追加したり削除したりして作った型を変えることができる。

In [ ]:
uselog['年月'].unique()
#numpy型(ndarray)
  #一度作った形を変えることができない。

#※「pandas」は「numpy」が土台。データの加工・計算する際には「numpy型(ndarray)」に自動で変換される。

#numpy ：「数値の塊」を操る。数学の天才。大量の数値を超高速で計算する。
#pandas：「データフレーム(表)の構造」を操る。例）並べ替え、抽出、結合、欠損値の穴埋め、CSVの読み書きなど

In [ ]:
predict_data = predict_data.dropna()
predict_data = predict_data.reset_index(drop=True)
predict_data.head()

#df.dropna()：欠損値(NaN)を含むデータをまとめて削除。
#df.reset_index()：すでに出来上がっているデータフレーム(表)に対する実行。
#df.reset_index(drop=True)：古い行番号(index)を削除して、「0から始まる新しい行番号(index)」を振り直す。
#df.reset_index(drop=False)：古い行番号(index)を「列(ラベル：index)」として残しつつ、「0から始まる新しい行番号(index)」を振り直す。

### ノック37：特徴となる変数を付与しよう

In [ ]:
customer.head()

In [ ]:
predict_data = pd.merge(predict_data, customer[['customer_id','start_date']], on='customer_id', how='left')
predict_data.head()

In [ ]:
predict_data['now_date'] = pd.to_datetime(predict_data['年月'], format='%Y%m')
predict_data['start_date'] = pd.to_datetime(predict_data['start_date'])

#pd.to_datetime：「文字(object)」として読み込まれた日付を、計算や加工ができる『日付専用データ(datetime)』に作り変える。
#to：～へ変換する。

#pd.to_datetimeの引数「format」：AIに日付(文字)の読み方を「%」で学習させる。
#日付(文字)の読み方：「%Y」西暦(4桁),「%y」西暦(2桁),「%m」月(2桁),「%d」日(2桁),「%H」時(2桁),「%M」分(2桁)

In [ ]:
from dateutil.relativedelta import relativedelta
predict_data['period'] = None

#from dateutil.relativedelta：『ライブラリ（dateutil）』に格納される『モジュール（relativedelta）』を指定する。
#import relativedelta：『モジュール（relativedelta）』に格納される『クラス（relativedelta）』を自分のプログラムに導入する。

#ライブラリ（dateutil）：日付や時刻の計算に特化したライブラリ。
#モジュール（relativedelta）：日付計算プログラムを格納したファイル。
#クラス（relativedelta）：日付計算AI。実際に日付計算を行う正体。

#None：値が存在しない。

In [ ]:
for i in range(len(predict_data)):
    delta = relativedelta(predict_data.loc[i, 'now_date'], predict_data.loc[i, 'start_date'])
    predict_data.loc[i,'period'] = delta.years*12 + delta.months
predict_data.head()

#len：データの件数を数える。Excelのcount関数と同じ役割。

#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[行番号,列番号]：整数で指定。整数を使って条件に合うデータを探す。

### ノック38：来月の利用回数予測モデルを作成しよう

In [ ]:
predict_data = predict_data.loc[predict_data['start_date']>=pd.to_datetime('20180401')]

#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[]：整数で指定。整数を使って条件に合うデータを探す。

#pd.to_datetime：「文字（object）」として読み込まれた日付を、計算や加工ができる『日付専用データ（datetime）』に作り変える命令。
#to：～へ変換する。

In [ ]:
from sklearn import linear_model
import sklearn.model_selection
model = linear_model.LinearRegression()

#ライブラリ（sklearn）：「教師あり学習」「教師なし学習」に100%特化したライブラリ。※強化学習は非対応。
#モジュール（linear_model）：「教師あり学習」のプログラムを格納したファイル。
#モジュール（model_selection）：AIの実力を測るプログラムを格納したファイル。
#クラス（LinearRegression）：線形回帰AI。バラバラなデータに「直線」を引いてルールを見つけるAI。教師あり学習の「回帰」を担当する。

#パッケージ：類似性のあるモジュールを格納したフォルダ。
#モジュール：プログラムを再利用するために、コードを格納したファイル。
#クラス：データ型。

In [ ]:
X = predict_data[['count_0','count_1','count_2','count_3','count_4','count_5','period']]
y = predict_data['count_pred']

# X ：説明変数（予測に使う変数）
# y ：目的変数（予測したい変数）

In [ ]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X,y, random_state=0)
model.fit(X_train, y_train)

#train_test_split()：学習用データと評価用データに分割する。※過学習状態を避けるため。
#過学習状態：学習用データに過剰適応することで、未知なデータに対応できなくなること。
#※無指定の場合、学習用データ(75%)、評価用データ(25%)に分割される。

#random_state：AIのランダム性(気の変わり)を抑える。
#random_state=0：毎回、「0」が名前に設定されたデータを出力。

#モデル.fit：AIの学習を開始する。

In [ ]:
print(model.score(X_train, y_train))
print(model.score(X_test, y_test))

#model.score(説明変数,目的変数)：「モデル(AI)の予測精度(0～1)↓」を測る。
# 1 ：完璧。過学習状態を疑う。
# 0.7～0.9 ：優秀。ビジネス(商売)で自信をもって使えるレベル。
# 0.5～0.6 ：実用圏内。傾向はつかめている。もっとデータを増やせば予測精度が上がるかも。
# 0.3以下 ：厳しい。「説明変数が足りない」か、「データのバラつきが大きくて直線で表すには無理がある」。
# 0 ：使い物にならない。平均値を答えるより外れている。やり直し！

### ノック39：モデルに寄与している変数を確認しよう

In [ ]:
#係数：そのデータが予測結果に与えている影響力。
#寄与：貢献

In [ ]:
coef = pd.DataFrame({'feature_names':X.columns, 'coefficient':model.coef_})
coef

#pd.DataFrame：データフレーム(表)を作成する。
#X.columns：変数名。データフレーム(表)の列ラベル。
#model.coef_：ライブラリ（sklearn）で定義された回帰モデルの係数(変数)。そのデータが予測結果に与えている影響力を数値化。

### ノック40：来月の利用回数を予測しよう

In [ ]:
x1 = [3, 4, 4, 6, 8, 7, 8]
x2 = [2, 2, 3, 3, 4, 6, 8]
x_pred = pd.DataFrame(data=[x1, x2],columns=['count_0','count_1','count_2','count_3','count_4','count_5','period'])
x_pred

In [ ]:
model.predict(x_pred)

#model.score(説明変数,目的変数)：「モデル(AI)の予測精度」を測る。
#model.predict(df)：AIで未来の数値を予測する。

In [ ]:
uselog_months.to_csv('uselog_ months.csv',index=False)

#df.to_csv：データをCSV形式で保存する。※分析結果をExcelで確認できるようにする。
#to：～へ変換する。

#index=True：行番号もファイルに書き出す(保存する)。
#index=False：行番号を捨てて、中身だけ保存する。